# Fine-tuning de RoBERTa biomédico-clínico para el NER de AI-Triage (paso 3.1)

**Modelo base:** [`PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`](https://huggingface.co/PlanTL-GOB-ES/roberta-base-biomedical-clinical-es) (Apache-2.0); réplica en `BSC-LT/roberta-base-biomedical-clinical-es`.

**Etiquetas:** `SINTOMA`, `ENFERMEDAD` y `FARMACO`, en esquema BIO. El modelo solo detecta menciones. Si una mención está negada, es un antecedente o es una alergia lo decide el pipeline después (`negation.py` y reglas de contexto).

| Etapa | Datos | Para qué |
| :--- | :--- | :--- |
| 1 | SympTEMIST + DisTEMIST + DrugTEMIST (CC BY 4.0), fusionados por documento; PharmaCoNER opcional | Aprender las tres entidades en casos clínicos escritos |
| 2 | Split `dev` del corpus propio (150 transcripciones, lotes 1 y 2) | Adaptar al habla transcrita de urgencias |

**Reglas de evaluación (no negociables):**
- El split `test` del corpus propio **no se usa aquí**. Se evalúa una sola vez en el paso 4.1.
- La etapa 2 se mide con **validación cruzada de 5 particiones sobre `dev`**. Se prueban varias configuraciones y se elige la de mayor F1 media; el modelo final se entrena con ella y con todo `dev`.
- Semilla fija (`SEED`) e hiperparámetros guardados en `training_config.json`.

**Entorno:** Colab o Kaggle con GPU T4. Activa la GPU antes de ejecutar (*Entorno de ejecución → Cambiar tipo → T4*). Duración estimada: entre 60 y 90 minutos la primera vez. Si reutilizas la etapa 1 de una corrida anterior (`REUSE_STAGE1`, sección 6), unos 20 minutos.

In [ ]:
# 1. Dependencias y GPU. Sin -q para ver cualquier error; el codigo funciona con transformers 4 y 5.
!pip install "datasets>=2.19" accelerate
import torch, transformers, datasets
print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU: activa una T4 antes de seguir')
SEED = 42

## 2. Código y corpus propio

El repositorio es privado, así que hay dos opciones:
- **A. Paquete (recomendado).** En tu equipo, ejecuta `python ml/ner/make_bundle.py` y sube `ml/ner/output/ner_bundle.zip` cuando la celda lo pida. En Kaggle, súbelo como *Dataset* y ajusta `BUNDLE`.
- **B. Clonar.** Guarda un token de GitHub de solo lectura en los secretos de Colab con el nombre `GITHUB_TOKEN`.

In [ ]:
import os, sys, zipfile
from pathlib import Path

WORK = Path('/content' if Path('/content').exists() else '/kaggle/working')
CODE = WORK / 'ner_code'
CODE.mkdir(parents=True, exist_ok=True)
BUNDLE = Path('/kaggle/input/ner-bundle/ner_bundle.zip')  # Kaggle: ruta del dataset subido

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    !git clone -q --depth 1 https://{token}@github.com/kevinJ1910/ai-triage.git {WORK}/repo
    for name, rel in {'ner_data.py': 'ml/ner/ner_data.py', 'train_ner.py': 'ml/ner/train_ner.py',
                      'error_analysis.py': 'ml/ner/error_analysis.py',
                      'corpus_entidades.json': 'backend/triage/benchmark/corpus_entidades.json'}.items():
        (CODE / name).write_bytes((WORK / 'repo' / rel).read_bytes())
else:
    if not BUNDLE.exists():
        from google.colab import files
        uploaded = files.upload()  # selecciona ner_bundle.zip
        BUNDLE = WORK / next(iter(uploaded))
    zipfile.ZipFile(BUNDLE).extractall(CODE)

sys.path.insert(0, str(CODE))
import ner_data, train_ner, error_analysis
print(sorted(p.name for p in CODE.iterdir()))

## 3. Corpus públicos (Zenodo, CC BY 4.0)

Se verifica el MD5 publicado por Zenodo. SympTEMIST ocupa unos 450 MB porque incluye el *background set* y traducciones, que no se usan. DrugTEMIST viene dentro del ZIP de MultiCardioNER (pista 2).

**FARMACO sale de DrugTEMIST.** Sus guías anotan medicamentos, como `medications` del corpus propio. PharmaCoNER también anota sustancias químicas, proteínas y genes. Se puede sumar con `USE_PHARMACONER = True`, pero entonces solo quedan los documentos presentes en los cuatro corpus.

In [ ]:
import hashlib, re, time, urllib.error, urllib.request

DATASETS = {
    'distemist': ('https://zenodo.org/api/records/7614764/files/distemist_zenodo.zip/content', 'bc84461ad07b59a94c79ca9c648952c2'),
    'pharmaconer': ('https://zenodo.org/api/records/4270158/files/pharmaconer.zip/content', 'e511314c4468c2655ae3355ce590a2c0'),
    'symptemist': ('https://zenodo.org/api/records/10635215/files/symptemist-complete_240208.zip/content', 'd31f61b68cf111cb96790301e5adca33'),
    'multicardioner': ('https://zenodo.org/api/records/11368861/files/multicardioner_train+dev+test+bg+mappings_240528.zip/content', '56055c807e2cc329ff5eeaf7e4f713ba'),
}
USE_PHARMACONER = False
DATA = WORK / 'datos'
DATA.mkdir(exist_ok=True)

def md5(path):
    digest = hashlib.md5()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download(url, target, attempts=6):
    # Descarga con reintentos y reanudacion (cabecera Range): Zenodo a veces corta los ZIP grandes.
    for attempt in range(1, attempts + 1):
        done = target.stat().st_size if target.exists() else 0
        headers = {'Range': f'bytes={done}-'} if done else {}
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as response:
                resumed = bool(done) and response.status == 206
                expected = int(response.headers.get('Content-Length', -1))
                expected = expected + done if resumed and expected >= 0 else expected
                with open(target, 'ab' if resumed else 'wb') as handle:
                    while chunk := response.read(1 << 20):
                        handle.write(chunk)
            if expected < 0 or target.stat().st_size >= expected:
                return
            print(f'  intento {attempt}: {target.stat().st_size} de {expected} bytes; se reanuda en 10 s')
        except urllib.error.HTTPError as err:
            if err.code == 416:  # el parcial ya esta completo (o corrupto): se empieza de cero
                target.unlink()
            print(f'  intento {attempt}: {err}')
        except Exception as err:
            print(f'  intento {attempt} interrumpido ({err}); se reanuda en 10 s')
        time.sleep(10)
    raise RuntimeError(f'No se pudo descargar {url} tras {attempts} intentos')


for name, (url, checksum) in DATASETS.items():
    archive = DATA / f'{name}.zip'
    for _ in range(3):
        if archive.exists() and md5(archive) == checksum:
            break
        print('Descargando', name)
        download(url, archive)
        if md5(archive) != checksum:
            print('  MD5 distinto: se descarga de nuevo desde cero')
            archive.unlink()
    assert archive.exists() and md5(archive) == checksum, f'MD5 incorrecto en {name}'
    if not (DATA / name).exists():
        zipfile.ZipFile(archive).extractall(DATA / name)
    print(name, 'OK')

In [ ]:
# 4. Localizar los ficheros. La estructura interna cambia entre versiones: se buscan por patrón y se muestran.
from collections import Counter
find = ner_data.find_files

dis_tsvs = [p for p in find(DATA / 'distemist', '*subtrack1*mentions*.tsv')]
pharma_dirs = sorted({p.parent for p in find(DATA / 'pharmaconer', '*.ann') if 'subtrack1' in str(p)})
sym_tsvs = [p for p in find(DATA / 'symptemist', '*.tsv')
            if not ner_data._EXCLUDED_DIRS.search(str(p)) and 'ner' in str(p).lower()
            and 'linking' not in str(p).lower() and 'norm' not in p.name.lower()]
# DrugTEMIST: carpetas brat de la pista de medicamentos; se excluyen las de ingles e italiano y el background.
LANG_DIR = re.compile(r'(^|[/_-])(en|it|english|italian)([/_-]|$)', re.IGNORECASE)
drug_dirs = sorted({p.parent for p in find(DATA / 'multicardioner', '*.ann')
                    if 'drug' in str(p.relative_to(DATA)).lower()
                    and not LANG_DIR.search(p.parent.relative_to(DATA).as_posix())
                    and not ner_data._EXCLUDED_DIRS.search(str(p))})

print('DisTEMIST TSV:', *dis_tsvs, sep='\n  ')
print('PharmaCoNER brat:', *pharma_dirs, sep='\n  ')
print('SympTEMIST TSV:', *sym_tsvs, sep='\n  ')
print('DrugTEMIST brat:', *drug_dirs, sep='\n  ')
assert dis_tsvs and sym_tsvs and drug_dirs, 'Revisa las rutas: algún corpus no se encontró'
assert pharma_dirs or not USE_PHARMACONER, 'PharmaCoNER no se encontró'

Revisa la salida anterior antes de seguir:
- **SympTEMIST:** deben aparecer solo los TSV de la subtarea de NER (`SINTOMA`), uno por split. Si se cuela otro TSV, por ejemplo uno de *linking* con códigos, quítalo de `sym_tsvs`.
- **DrugTEMIST:** deben aparecer solo las carpetas brat de medicamentos en español, de entrenamiento y test. Si se cuela una de CardioCCC (cardiología), puedes dejarla: usa las mismas guías.

In [ ]:
# 5. Cargar y fusionar por documento. Se usan todos los splits oficiales con anotación gold (train, dev y test)
#    porque aquí no se reportan métricas sobre ellos: la validación de la etapa 1 es un 10 % aparte.
stats = Counter()

def merge_splits(*parts):
    merged = {}
    for part in parts:
        merged.update(part)
    return merged

symptemist = merge_splits(*[
    ner_data.load_tsv_corpus(tsv, ner_data.pick_text_dir(DATA / 'symptemist', ner_data.read_mentions_tsv(tsv)),
                             'symptemist', stats) for tsv in sym_tsvs])
distemist = merge_splits(*[
    ner_data.load_tsv_corpus(tsv, ner_data.pick_text_dir(tsv.parent.parent, ner_data.read_mentions_tsv(tsv)),
                             'distemist', stats) for tsv in dis_tsvs])
drugtemist = merge_splits(*[ner_data.load_brat_corpus(d, 'drugtemist', stats) for d in drug_dirs])
corpora = {'symptemist': symptemist, 'distemist': distemist, 'drugtemist': drugtemist}
if USE_PHARMACONER:
    corpora['pharmaconer'] = merge_splits(*[ner_data.load_brat_corpus(d, 'pharmaconer', stats) for d in pharma_dirs])

for name, corpus in corpora.items():
    print(f'{name}: {len(corpus)} documentos')

public_docs = ner_data.merge_corpora(list(corpora.values()), required=set(corpora), stats=stats)
print('Fusionados con todas las capas:', ner_data.corpus_summary(public_docs))
print(dict(stats))
assert len(public_docs) > 300, 'Muy pocos documentos comunes: revisa las rutas y los nombres de fichero'

En `stats`, `documento_parcial_descartado` cuenta los documentos a los que les falta alguno de los corpus usados, y `drugtemist:otro_idioma` los ficheros en inglés o italiano que se ignoraron. No se usan porque les faltaría alguna capa de anotación. `texto_distinto` y `span_no_reubicado` indican diferencias de versión entre corpus; si son muchos, conviene revisarlos antes de entrenar.

## 6. Etapa 1

Si ya tienes `etapa1_modelo.zip` de una corrida anterior, pon `REUSE_STAGE1 = True` y súbelo cuando se pida. Así te ahorras la descarga de los corpus y unos 40 minutos de entrenamiento, y puedes saltarte las secciones 3, 4 y 5. Úsalo para iterar sobre la etapa 2, por ejemplo después de revisar el corpus propio.

**Cambio respecto a la primera corrida:** 5 épocas en lugar de 3. Se evalúa al final de cada época y se queda el mejor checkpoint, así que más épocas no pueden empeorar el modelo elegido.

In [ ]:
# 6. Etapa 1: corpus públicos. Validación con un 10 % de documentos apartados (para elegir el checkpoint).
import json, shutil
from transformers import set_seed
set_seed(SEED)

REUSE_STAGE1 = False
STAGE1 = dict(epochs=5, learning_rate=5e-5, batch_size=16, max_length=512, stride=128, seed=SEED)
STAGE1_DIR = WORK / 'etapa1_modelo'
tokenizer = train_ner.load_tokenizer()

if REUSE_STAGE1:
    from google.colab import files
    uploaded = files.upload()  # selecciona etapa1_modelo.zip
    zipfile.ZipFile(WORK / next(iter(uploaded))).extractall(STAGE1_DIR)
    stage1_info = json.loads((STAGE1_DIR / 'etapa1_info.json').read_text(encoding='utf-8'))
    print('Etapa 1 reutilizada:', stage1_info['metricas']['micro'])
else:
    stage1_train, stage1_val = ner_data.holdout(public_docs, fraction=0.1, seed=SEED)
    print('train', len(stage1_train), '| val', len(stage1_val))
    trainer1 = train_ner.train(train_ner.new_model(), tokenizer, stage1_train, stage1_val, str(WORK / 'etapa1'), **STAGE1)
    predicted = train_ner.predict_spans(trainer1.model, tokenizer, stage1_val)
    gold = {d.doc_id: d.spans for d in stage1_val}
    stage1_info = {
        'metricas': ner_data.span_prf(gold, predicted),
        'metricas_parciales': ner_data.span_prf_partial(gold, predicted),
        'config': {**STAGE1, 'documentos_train': len(stage1_train), 'documentos_val': len(stage1_val),
                   'corpus_usados': sorted(corpora),
                   'corpus': {name: url for name, (url, _) in DATASETS.items()}, 'estadisticas_fusion': dict(stats)},
    }
    trainer1.save_model(str(STAGE1_DIR))
    tokenizer.save_pretrained(str(STAGE1_DIR))
    (STAGE1_DIR / 'etapa1_info.json').write_text(json.dumps(stage1_info, ensure_ascii=False, indent=2), encoding='utf-8')
    (STAGE1_DIR / 'etapa1_predicciones.json').write_text(
        json.dumps(train_ner.predictions_payload(stage1_val, predicted), ensure_ascii=False), encoding='utf-8')
    # Guárdalo: con él, la próxima corrida empieza directamente en la etapa 2.
    shutil.make_archive(str(WORK / 'etapa1_modelo'), 'zip', STAGE1_DIR)
    del trainer1
    torch.cuda.empty_cache()
    print('Etapa 1, validación. Exacta:', stage1_info['metricas']['micro'])
    print('Etapa 1, validación. Parcial:', stage1_info['metricas_parciales']['micro'])
    try:
        from google.colab import files
        files.download(str(WORK / 'etapa1_modelo.zip'))
    except Exception:
        print('Descarga etapa1_modelo.zip desde el panel de archivos.')
stage1_model = lambda: train_ner.new_model(str(STAGE1_DIR))

In [ ]:
# 7. Transferencia sin adaptar: modelo de la etapa 1 sobre las transcripciones de dev del corpus propio.
own_dev = ner_data.corpus_to_documents(CODE / 'corpus_entidades.json', split='dev')
print('dev propio:', ner_data.corpus_summary(own_dev))
model1 = stage1_model().to('cuda' if torch.cuda.is_available() else 'cpu')
zero_shot_pred = train_ner.predict_spans(model1, tokenizer, own_dev)
dev_gold = {d.doc_id: d.spans for d in own_dev}
zero_shot = ner_data.span_prf(dev_gold, zero_shot_pred)
zero_shot_partial = ner_data.span_prf_partial(dev_gold, zero_shot_pred)
zero_shot_payload = train_ner.predictions_payload(own_dev, zero_shot_pred)
del model1
for label in zero_shot:
    print(f'{label:<11} exacta {zero_shot[label]["f1"]} | parcial {zero_shot_partial[label]["f1"]}')

## 8. Etapa 2: rejilla de configuraciones con validación cruzada

Cada configuración se mide con las mismas 5 particiones de `dev`, y se elige la de mayor F1 exacta media. Se comparan 10, 20 y 30 épocas. El aumento de datos (copias en minúsculas y sin puntuación) se probó en la corrida del 17-10-2026 y no aportó, así que ya no está en la rejilla. Cada configuración tarda pocos minutos.

**Para la tesis:** como la configuración se elige con esta misma validación cruzada, la cifra de la ganadora es algo optimista. La cifra que se reporta como resultado es la de `test` (paso 4.1). La tabla de la rejilla sirve como ablación.

In [ ]:
# 8. Etapa 2: rejilla de configuraciones, validación cruzada de 5 particiones sobre dev.
STAGE2_BASE = dict(batch_size=8, max_length=512, stride=128)
# El aumento de datos (minúsculas y sin puntuación) se probó el 17-10-2026 y no aportó: 0,738 frente a 0,730.
# Para volver a probarlo: 'aumento': dict(epochs=10, learning_rate=2e-5, augment=('minusculas', 'sin_puntuacion')).
STAGE2_GRID = {
    'base': dict(epochs=10, learning_rate=2e-5, augment=()),
    'mas_epocas': dict(epochs=20, learning_rate=3e-5, augment=()),
    'epocas_30': dict(epochs=30, learning_rate=3e-5, augment=()),
}
grid_results = {}
for name, options in STAGE2_GRID.items():
    print(f'--- {name}: {options}')
    grid_results[name] = train_ner.cross_validate(stage1_model, tokenizer, own_dev, str(WORK / f'cv_{name}'),
                                                  k=5, seed=SEED, **STAGE2_BASE, **options)

COLUMNS = ('exacta', 'parcial', 'exacta_sin_posproceso')
print(f"\n{'configuración':<14}{'exacta':>16}{'parcial':>16}{'sin posproceso':>16}")
for name, result in grid_results.items():
    print(f"{name:<14}" + ''.join(f"{result[k]['micro']['f1']['media']:>9} ± {result[k]['micro']['f1']['desviacion']:<5}"
                                  for k in COLUMNS))
best_name = max(grid_results, key=lambda n: grid_results[n]['exacta']['micro']['f1']['media'])
best = grid_results[best_name]
STAGE2 = {**STAGE2_BASE, **STAGE2_GRID[best_name], 'seed': SEED}
print('Elegida:', best_name)
print(json.dumps(best['exacta'], indent=2, ensure_ascii=False))

In [ ]:
# 9. Análisis de errores de la configuración elegida: límites, etiqueta, omitidos y espurios.
#    Si dominan los errores de límites, conviene unificar la guía de anotación antes de reentrenar.
error_report = error_analysis.error_report(best['predicciones'])
error_analysis._print_errors(error_report)

In [ ]:
# 10. Modelo final: etapa 1 + todo dev con la configuración elegida. Se guarda con métricas, configuración y análisis.
import datetime

final_docs = ner_data.augment_documents(own_dev, STAGE2['augment']) if STAGE2['augment'] else own_dev
final_trainer = train_ner.train(stage1_model(), tokenizer, final_docs, None, str(WORK / 'etapa2_final'),
                                **{k: v for k, v in STAGE2.items() if k != 'augment'})

summary_of = lambda r: {k: r[k] for k in ('exacta', 'parcial', 'exacta_sin_posproceso')}
metrics = {
    'etapa1_validacion_spans': stage1_info['metricas'],
    'etapa1_validacion_parcial': stage1_info.get('metricas_parciales'),
    'transferencia_sin_adaptar_dev': zero_shot,
    'transferencia_sin_adaptar_dev_parcial': zero_shot_partial,
    'etapa2_validacion_cruzada_dev_5_particiones': best['exacta'],
    'etapa2_validacion_cruzada_parcial': best['parcial'],
    'etapa2_configuracion_elegida': best_name,
    'etapa2_rejilla': {name: summary_of(r) for name, r in grid_results.items()},
    'errores_etapa2': {k: error_report[k] for k in ('reparto_de_errores', 'errores_por_tipo', 'patrones_de_limites',
                                                    'confusiones_de_etiqueta')},
    'nota': ('El split test del corpus propio no se ha usado; se evalua una vez en el paso 4.1. La configuracion de '
             'la etapa 2 se eligio con la misma validacion cruzada, asi que su cifra es algo optimista.'),
}
config = {
    'modelo_base': train_ner.BASE_MODEL,
    'etiquetas': ner_data.LABELS,
    'etapa1': stage1_info['config'],
    'etapa2': {**STAGE2, 'augment': list(STAGE2['augment']), 'documentos': len(own_dev),
               'documentos_entrenamiento': len(final_docs), 'particiones': 5, 'rejilla': STAGE2_GRID},
    'semilla': SEED,
    'fecha': datetime.date.today().isoformat(),
    'versiones': {'torch': torch.__version__, 'transformers': transformers.__version__},
}
OUTPUT = WORK / 'ai_triage_ner_roberta'
train_ner.save_artifacts(final_trainer, tokenizer, str(OUTPUT), metrics, config)
ANALYSIS = OUTPUT / 'analisis'
ANALYSIS.mkdir(exist_ok=True)
for name, result in grid_results.items():
    (ANALYSIS / f'cv_predicciones_{name}.json').write_text(json.dumps(result['predicciones'], ensure_ascii=False),
                                                         encoding='utf-8')
(ANALYSIS / 'cv_predicciones.json').write_text(json.dumps(best['predicciones'], ensure_ascii=False), encoding='utf-8')
(ANALYSIS / 'transferencia_predicciones.json').write_text(json.dumps(zero_shot_payload, ensure_ascii=False),
                                                         encoding='utf-8')
if (STAGE1_DIR / 'etapa1_predicciones.json').exists():
    shutil.copy(STAGE1_DIR / 'etapa1_predicciones.json', ANALYSIS / 'etapa1_predicciones.json')
print(sorted(p.name for p in OUTPUT.iterdir()), sorted(p.name for p in ANALYSIS.iterdir()))

## 11. Exportar el modelo (nunca a git)

Hay dos opciones:
- **ZIP**, para adjuntarlo a una *release* de GitHub o descargarlo. Ejecuta la siguiente celda.
- **Hugging Face Hub, repositorio privado.** Guarda un token con permiso de escritura en los secretos de Colab con el nombre `HF_TOKEN`, pon `PUSH_TO_HUB = True` y ajusta `HUB_REPO`.

Después, en el backend, `NER_MODEL_PATH` apuntará a la carpeta descomprimida (paso 3.2).

In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / 'ai_triage_ner_roberta'), 'zip', OUTPUT)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    print('En Kaggle: descarga el ZIP desde la pestaña Output.')

PUSH_TO_HUB = False
HUB_REPO = 'TU_USUARIO/ai-triage-ner-roberta'
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    from google.colab import userdata
    api = HfApi(token=userdata.get('HF_TOKEN'))
    api.create_repo(HUB_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(OUTPUT), repo_id=HUB_REPO)
    print('Subido a', HUB_REPO, '(privado)')

## Qué reportar en la tesis

- **Etapa 1:** F1 por spans exactos en la validación apartada. Mide la tarea en texto clínico escrito.
- **Transferencia sin adaptar sobre dev:** cuánto se pierde al pasar de texto escrito a habla transcrita.
- **Etapa 2:** media ± desviación de la validación cruzada de 5 particiones en dev, exacta y parcial, y la tabla de la rejilla como ablación (base, aumento de datos, más épocas).
- **Análisis de errores:** reparto entre límites, etiqueta, omitidos y espurios (`metrics.json` y la carpeta `analisis/`). En local: `python ml/ner/error_analysis.py errores ml/ner/output/ai_triage_ner_roberta/analisis/cv_predicciones.json`.
- **Cifra principal:** la de `test`, que se calcula una sola vez en el paso 4.1 con el sistema híbrido completo.

Hay varias limitaciones que conviene declarar:
- El corpus propio es pequeño y está anotado con asistencia de IA, con revisión humana pendiente.
- La etiqueta ENFERMEDAD agrupa diagnósticos del episodio y antecedentes.
- Los corpus públicos son casos clínicos escritos, no diálogos.